### imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np

from torch_geometric.nn import RGCNConv

### Load Graph

In [2]:
edge_index = np.load(
    r"C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx\data\ml_ready\edge_index.npy"
)

edge_type = np.load(
    r"C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx\data\ml_ready\edge_type.npy"
)

edge_index = torch.tensor(
    edge_index,
    dtype=torch.long
)

edge_type = torch.tensor(
    edge_type,
    dtype=torch.long
)

print(edge_index.shape)
print(edge_type.shape)

torch.Size([2, 15573])
torch.Size([15573])


### Graph Information

In [3]:
num_nodes = 7502

num_relations = 5

embedding_dim = 128

### Define R-GCN

In [4]:
class RGCN(nn.Module):

    def __init__(
        self,
        num_nodes,
        num_relations,
        embedding_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            num_nodes,
            embedding_dim
        )

        self.conv1 = RGCNConv(
            embedding_dim,
            128,
            num_relations
        )

        self.conv2 = RGCNConv(
            128,
            128,
            num_relations
        )

    def forward(
        self,
        edge_index,
        edge_type
    ):

        x = self.embedding.weight

        x = self.conv1(
            x,
            edge_index,
            edge_type
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index,
            edge_type
        )

        return x

### Create Model

In [5]:
model = RGCN(
    num_nodes,
    num_relations,
    embedding_dim
)

model

RGCN(
  (embedding): Embedding(7502, 128)
  (conv1): RGCNConv(128, 128, num_relations=5)
  (conv2): RGCNConv(128, 128, num_relations=5)
)

### Test Forward Pass

In [6]:
with torch.no_grad():

    embeddings = model(
        edge_index,
        edge_type
    )

print(
    embeddings.shape
)

torch.Size([7502, 128])


---------------

## TRAIN R-GCN

### Positive Edges

In [7]:
positive_edges = edge_index.t()

print(
    positive_edges.shape
)

torch.Size([15573, 2])


### Negative Edges

In [8]:
num_edges = positive_edges.shape[0]

negative_edges = torch.randint(
    0,
    num_nodes,
    (num_edges, 2)
)

print(
    negative_edges.shape
)

torch.Size([15573, 2])


### Labels

In [9]:
positive_labels = torch.ones(
    num_edges
)

negative_labels = torch.zeros(
    num_edges
)

### Combine

In [10]:
all_edges = torch.cat(
    [
        positive_edges,
        negative_edges
    ],
    dim=0
)

all_labels = torch.cat(
    [
        positive_labels,
        negative_labels
    ],
    dim=0
)

print(all_edges.shape)

print(all_labels.shape)

torch.Size([31146, 2])
torch.Size([31146])


### Link Predictor

In [11]:
class LinkPredictor(nn.Module):

    def __init__(
        self,
        embedding_dim
    ):

        super().__init__()

        self.linear = nn.Linear(
            embedding_dim * 2,
            1
        )

    def forward(
        self,
        source,
        target
    ):

        x = torch.cat(
            [
                source,
                target
            ],
            dim=1
        )

        return self.linear(x)

### Create Predictor

In [12]:
predictor = LinkPredictor(
    128
)

### Optimizer

In [13]:
optimizer = torch.optim.Adam(

    list(model.parameters())
    +
    list(predictor.parameters()),

    lr=0.001
)

criterion = nn.BCEWithLogitsLoss()

### Training Loop

In [14]:
epochs = 20

for epoch in range(epochs):

    optimizer.zero_grad()

    embeddings = model(
        edge_index,
        edge_type
    )

    src = embeddings[
        all_edges[:,0]
    ]

    dst = embeddings[
        all_edges[:,1]
    ]

    pred = predictor(
        src,
        dst
    ).squeeze()

    loss = criterion(
        pred,
        all_labels
    )

    loss.backward()

    optimizer.step()

    print(
        f"Epoch {epoch+1} "
        f"Loss: {loss.item():.4f}"
    )

Epoch 1 Loss: 0.6713
Epoch 2 Loss: 0.5675
Epoch 3 Loss: 0.5003
Epoch 4 Loss: 0.4546
Epoch 5 Loss: 0.4199
Epoch 6 Loss: 0.3911
Epoch 7 Loss: 0.3660
Epoch 8 Loss: 0.3436
Epoch 9 Loss: 0.3239
Epoch 10 Loss: 0.3065
Epoch 11 Loss: 0.2913
Epoch 12 Loss: 0.2779
Epoch 13 Loss: 0.2659
Epoch 14 Loss: 0.2550
Epoch 15 Loss: 0.2452
Epoch 16 Loss: 0.2365
Epoch 17 Loss: 0.2287
Epoch 18 Loss: 0.2217
Epoch 19 Loss: 0.2153
Epoch 20 Loss: 0.2094


### Evaluate R-GCN

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

### Eval model

In [16]:
model.eval()

predictor.eval()

LinkPredictor(
  (linear): Linear(in_features=256, out_features=1, bias=True)
)

### Generate predictions

In [17]:
with torch.no_grad():

    embeddings = model(
        edge_index,
        edge_type
    )

    src = embeddings[
        all_edges[:,0]
    ]

    dst = embeddings[
        all_edges[:,1]
    ]

    logits = predictor(
        src,
        dst
    ).squeeze()

    probs = torch.sigmoid(
        logits
    )

### Convert To Labels

In [18]:
pred_labels = (
    probs > 0.5
).int()

### convert to numpy

In [19]:
y_true = all_labels.numpy()

y_pred = pred_labels.numpy()

y_prob = probs.numpy()

### Metrics

In [20]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred
)

recall = recall_score(
    y_true,
    y_pred
)

f1 = f1_score(
    y_true,
    y_pred
)

auc = roc_auc_score(
    y_true,
    y_prob
)

### Results

In [21]:
print("Accuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1 Score :", f1)

print("AUC      :", auc)

Accuracy : 0.9265074166827201
Precision: 0.891073951954781
Recall   : 0.9718101842933282
F1 Score : 0.9296925392388734
AUC      : 0.9729420348265718


### Save the R-GCN Model

In [23]:
from pathlib import Path

project = Path(
    r"C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx"
)

model_dir = project / "models"
model_dir.mkdir(exist_ok=True)

torch.save(
    model.state_dict(),
    model_dir / "rgcn_model.pth"
)

torch.save(
    predictor.state_dict(),
    model_dir / "rgcn_predictor.pth"
)

print("saved")

saved


#### Generate Node Embeddings

In [24]:
model.eval()

with torch.no_grad():

    node_embeddings = model(
        edge_index,
        edge_type
    )

print(node_embeddings.shape)

torch.Size([7502, 128])


### Load node2id

In [25]:
import pickle

with open(
    project / "data" / "ml_ready" / "node2id.pkl",
    "rb"
) as f:

    node2id = pickle.load(f)

### Create Reverse Mapping

In [26]:
id2node = {
    v:k
    for k,v in node2id.items()
}

print(len(id2node))

7502


### Inspect Node Types

In [27]:
sample = list(node2id.keys())[:20]

sample

['10Web:Form Maker by 10Web',
 '10web:Form Maker by 10Web – Mobile-Friendly Drag & Drop Contact Form Builder',
 '2download:2Download Connector for 2DL Hosted Checkout',
 '404-redirection-manager:404 Redirection Manager',
 '@remix-run:server-runtime',
 'A WP Life:Webenvo',
 'AA-Team:Premium Age Verification / Restriction for WordPress',
 'ACPT:ACPT (Pro) - Custom Post Types Plugin for WordPress',
 'ANSSI:DFIR-ORC',
 'AOMEI:Backupper',
 'AOMEI:Dynamic Disk Manager',
 'AOMEI:Partition Assistant',
 'ASUS:Armoury Crate',
 'AVer:PTC115',
 'AVer:PTC115+',
 'AVer:PTC500+',
 'AVer:PTC500S',
 'AVideo:AVideo',
 'AWS:Kiro IDE',
 'AWS:bedrock-agentcore']

In [28]:
types = {}

for node in node2id.keys():

    prefix = node.split("-")[0] if "-" in node else node.split(":")[0]

    types[prefix] = types.get(prefix, 0) + 1

for k,v in sorted(types.items(), key=lambda x:x[1], reverse=True)[:30]:
    print(k, v)

CVE 2024
Mitsubishi Electric Corporation:Room Air Conditioners (outside Japan) MSZ 1206
Mitsubishi Electric Corporation:Room Air Conditioners (for Japan) MSZ 1115
CAPEC 565
CWE 423
Oracle Corporation 61
ThemeREX 58
Mitsubishi Electric Corporation:Refrigerators (for Japan) MR 47
Mitsubishi Electric Corporation:Room Air Conditioners (outside Japan) MFZ 28
Red Hat 19
Unknown 16
Mitsubishi Electric Corporation:Wireless LAN Adapters for Room Air Conditioners (outside Japan) MAC 14
Elated 13
Mitsubishi Electric Corporation:Heat Pump Water Heaters / HEMS 13
Mitsubishi Electric Corporation:Wireless LAN Adapters for Room Air Conditioners (for Japan) HM 12
Dell 11
IBM 11
Mikado 11
Zyxel:GS1900 10
Microsoft 9
Mitsubishi Electric Corporation:Adapters for Airflow Ventilation Systems, Heat Pump Chilled/Hot Water Systems, and Ventilation/Air 9
SteeltoeOSS 9
Apache Software Foundation 7
CRM Perks 7
Mitsubishi Electric Corporation:Room Air Conditioners (outside Japan) MSXY 7
Select 7
TP 7
Canonical 6
E

In [29]:
cve_nodes = [n for n in node2id if str(n).startswith("CVE")]
cwe_nodes = [n for n in node2id if str(n).startswith("CWE")]
capec_nodes = [n for n in node2id if str(n).startswith("CAPEC")]
attack_nodes = [n for n in node2id if str(n).startswith("T")]

print("CVE:", len(cve_nodes))
print("CWE:", len(cwe_nodes))
print("CAPEC:", len(capec_nodes))
print("ATTACK:", len(attack_nodes))

CVE: 2024
CWE: 423
CAPEC: 565
ATTACK: 921


In [32]:
import pandas as pd

cve_cwe = pd.read_csv(project / "data" / "processed" / "cve_cwe_edges.csv")
cwe_capec = pd.read_csv(project / "data" / "processed" / "cwe_capec_edges.csv")
capec_attack = pd.read_csv(project / "data" / "processed" / "capec_attack_edges.csv")
attack_mitigation = pd.read_csv(project / "data" / "processed" / "attack_mitigation_edges.csv")
software_edges = pd.read_csv(project / "data" / "processed" / "software_edges.csv")

all_edges_df = pd.concat([
    cve_cwe,
    cwe_capec,
    capec_attack,
    attack_mitigation,
    software_edges
])

print(all_edges_df["relation"].value_counts())

relation
maps_to         5845
affects         4772
has_weakness    2296
mitigated_by    1448
mapped_to       1212
Name: count, dtype: int64


**Create a graph object from all edges**

In [33]:
import networkx as nx

G = nx.DiGraph()

for _, row in all_edges_df.iterrows():

    G.add_edge(
        row["source"],
        row["target"],
        relation=row["relation"]
    )

print(
    G.number_of_nodes(),
    G.number_of_edges()
)

7502 15573


### verify

In [34]:
cve_nodes = [
    n for n in G.nodes
    if str(n).startswith("CVE")
]

cve_nodes[:5]

['CVE-2026-12212',
 'CVE-2026-12213',
 'CVE-2026-12214',
 'CVE-2026-12216',
 'CVE-2026-12217']

### Picking one CVE

In [35]:
start_cve = "CVE-2026-12212"

### Find neighboring CWE nodes

In [36]:
cwe_neighbors = []

for neighbor in G.successors(start_cve):

    relation = G[start_cve][neighbor]["relation"]

    if relation == "has_weakness":

        cwe_neighbors.append(neighbor)

cwe_neighbors[:10]

['CWE-266', 'CWE-284']

### CWE → CAPEC
- see which attack patterns are associated with those weaknesses

In [37]:
capec_neighbors = []

for cwe in cwe_neighbors:

    if cwe in G:

        for neighbor in G.successors(cwe):

            relation = G[cwe][neighbor]["relation"]

            if relation == "mapped_to":

                capec_neighbors.append(neighbor)

capec_neighbors[:20]

['CAPEC-19',
 'CAPEC-441',
 'CAPEC-478',
 'CAPEC-479',
 'CAPEC-502',
 'CAPEC-503',
 'CAPEC-536',
 'CAPEC-546',
 'CAPEC-550',
 'CAPEC-551',
 'CAPEC-552',
 'CAPEC-556',
 'CAPEC-558',
 'CAPEC-562',
 'CAPEC-563',
 'CAPEC-564',
 'CAPEC-578']

### CAPEC → ATT&CK
- discover the ATT&CK techniques connected to these attack patterns.

In [38]:
attack_neighbors = []

for capec in capec_neighbors:

    if capec in G:

        for neighbor in G.successors(capec):

            relation = G[capec][neighbor]["relation"]

            if relation == "maps_to":

                attack_neighbors.append(neighbor)

attack_neighbors = list(set(attack_neighbors))

print("ATT&CK techniques:", len(attack_neighbors))

attack_neighbors[:30]

ATT&CK techniques: 164


['T1037',
 'T1036.004',
 'T1602.002',
 'T1583.006',
 'T1021.007',
 'T1084',
 'T1485.001',
 'T1555.002',
 'T1526',
 'T1518.002',
 'T1213.004',
 'T1592.002',
 'T1555.004',
 'T1588.004',
 'T1684.002',
 'T1039',
 'T1037.003',
 'T1562.001',
 'T1195.001',
 'T1003.002',
 'T1546.003',
 'T1176',
 'T1602',
 'T1685',
 'T1130',
 'T1014',
 'T1156',
 'T1127',
 'T1559.003',
 'T1543.005']

### ATT&CK → Mitigation
- complete the full chain first.

In [39]:
mitigations = []

for attack in attack_neighbors:

    if attack in G:

        for neighbor in G.successors(attack):

            relation = G[attack][neighbor]["relation"]

            if relation == "mitigated_by":

                mitigations.append(neighbor)

mitigations = list(set(mitigations))

print("Mitigations:", len(mitigations))

mitigations[:30]

Mitigations: 39


['Limit Access to Resource Over Network',
 'Network Segmentation',
 'Execution Prevention',
 'Data Backup',
 'Restrict Web-Based Content',
 'Password Policies',
 'Threat Intelligence Program',
 'Restrict File and Directory Permissions',
 'Code Signing',
 'Credential Access Protection',
 'Privileged Account Management',
 'Restrict Registry Permissions',
 'Antivirus/Antimalware',
 'Software Configuration',
 'Active Directory Configuration',
 'Boot Integrity',
 'Operating System Configuration',
 'Application Isolation and Sandboxing',
 'User Account Control',
 'Remote Data Storage',
 'Encrypt Sensitive Information',
 'Network Intrusion Prevention',
 'Disable or Remove Feature or Program',
 'Vulnerability Scanning',
 'Privileged Process Integrity',
 'Pre-compromise',
 'User Account Management',
 'Exploit Protection',
 'Audit',
 'Do Not Mitigate']

---------

### explain_cve()

In [40]:
def explain_cve(cve_id):

    if cve_id not in G:
        print("CVE not found")
        return

    cwes = []
    capecs = []
    attacks = []
    mitigations = []

    # CVE -> CWE
    for cwe in G.successors(cve_id):

        if G[cve_id][cwe]["relation"] == "has_weakness":

            cwes.append(cwe)

            # CWE -> CAPEC
            for capec in G.successors(cwe):

                if G[cwe][capec]["relation"] == "mapped_to":

                    capecs.append(capec)

                    # CAPEC -> ATTACK
                    for attack in G.successors(capec):

                        if G[capec][attack]["relation"] == "maps_to":

                            attacks.append(attack)

                            # ATTACK -> Mitigation
                            for mitigation in G.successors(attack):

                                if G[attack][mitigation]["relation"] == "mitigated_by":

                                    mitigations.append(mitigation)

    print("\nCVE:", cve_id)

    print("\nWeaknesses:")
    print(sorted(set(cwes))[:20])

    print("\nAttack Patterns:")
    print(sorted(set(capecs))[:20])

    print("\nATT&CK Techniques:")
    print(sorted(set(attacks))[:20])

    print("\nMitigations:")
    print(sorted(set(mitigations))[:20])

In [41]:
explain_cve("CVE-2026-12212")


CVE: CVE-2026-12212

Weaknesses:
['CWE-266', 'CWE-284']

Attack Patterns:
['CAPEC-19', 'CAPEC-441', 'CAPEC-478', 'CAPEC-479', 'CAPEC-502', 'CAPEC-503', 'CAPEC-536', 'CAPEC-546', 'CAPEC-550', 'CAPEC-551', 'CAPEC-552', 'CAPEC-556', 'CAPEC-558', 'CAPEC-562', 'CAPEC-563', 'CAPEC-564', 'CAPEC-578']

ATT&CK Techniques:
['T1001.003', 'T1003.002', 'T1004', 'T1007', 'T1014', 'T1016', 'T1017', 'T1021', 'T1021.002', 'T1021.006', 'T1021.007', 'T1023', 'T1027.002', 'T1027.016', 'T1028', 'T1031', 'T1035', 'T1036.004', 'T1036.007', 'T1037']

Mitigations:
['Active Directory Configuration', 'Antivirus/Antimalware', 'Application Developer Guidance', 'Application Isolation and Sandboxing', 'Audit', 'Behavior Prevention on Endpoint', 'Boot Integrity', 'Code Signing', 'Credential Access Protection', 'Data Backup', 'Data Loss Prevention', 'Disable or Remove Feature or Program', 'Do Not Mitigate', 'Encrypt Sensitive Information', 'Execution Prevention', 'Exploit Protection', 'Filter Network Traffic', 'Limit

----------

### R-GCN Discovery

In [42]:
node_embeddings.shape

torch.Size([7502, 128])

In [43]:
print(type(node_embeddings))

<class 'torch.Tensor'>


### Select Candidate Nodes

In [44]:
cve_nodes = [
    n for n in node2id
    if n.startswith("CVE-")
]

attack_nodes = [
    n for n in node2id
    if n.startswith("T")
]

print("CVEs:", len(cve_nodes))
print("ATT&CK:", len(attack_nodes))

CVEs: 2024
ATT&CK: 921


### One CVE

In [45]:
target_cve = "CVE-2026-12212"

print(target_cve in node2id)

True


### Score Every ATT&CK Technique

In [46]:
model.eval()
predictor.eval()

cve_idx = node2id[target_cve]

predictions = []

with torch.no_grad():

    cve_embedding = node_embeddings[
        cve_idx
    ].unsqueeze(0)

    for attack in attack_nodes:

        attack_idx = node2id[attack]

        attack_embedding = node_embeddings[
            attack_idx
        ].unsqueeze(0)

        score = torch.sigmoid(
            predictor(
                cve_embedding,
                attack_embedding
            )
        ).item()

        predictions.append(
            (
                attack,
                score
            )
        )

### Rank Results

In [47]:
predictions = sorted(
    predictions,
    key=lambda x: x[1],
    reverse=True
)

predictions[:20]

[('Threat Intelligence Program', 0.9675381779670715),
 ('T1020', 0.960335373878479),
 ('T1045', 0.9577574729919434),
 ('T1218.002', 0.9564635157585144),
 ('T1522', 0.9529238939285278),
 ('T1668', 0.9506741762161255),
 ('T1048', 0.9388517141342163),
 ('T1006', 0.9342100024223328),
 ('T1552.005', 0.932371199131012),
 ('T1684.002', 0.9310784339904785),
 ('T1499.001', 0.930199921131134),
 ('T1497.002', 0.9292735457420349),
 ('T1196', 0.9277694821357727),
 ('T1020.001', 0.9249854683876038),
 ('T1592.002', 0.9244982004165649),
 ('T1562.011', 0.9229199886322021),
 ('T1061', 0.9220420122146606),
 ('T1205', 0.9215130805969238),
 ('T1071.003', 0.9176769256591797),
 ('T1492', 0.9148496985435486)]

In [48]:
import re

attack_nodes = [
    n for n in node2id
    if re.match(r"^T\d+", n)
]

print(len(attack_nodes))
print(attack_nodes[:10])

806
['T1001', 'T1001.001', 'T1001.002', 'T1001.003', 'T1003', 'T1003.001', 'T1003.002', 'T1003.003', 'T1003.004', 'T1003.005']


In [49]:
predictions = []

with torch.no_grad():

    cve_embedding = node_embeddings[cve_idx].unsqueeze(0)

    for attack in attack_nodes:

        attack_idx = node2id[attack]

        attack_embedding = node_embeddings[
            attack_idx
        ].unsqueeze(0)

        score = torch.sigmoid(
            predictor(
                cve_embedding,
                attack_embedding
            )
        ).item()

        predictions.append(
            (
                attack,
                score
            )
        )

predictions = sorted(
    predictions,
    key=lambda x: x[1],
    reverse=True
)

predictions[:20]

[('T1020', 0.960335373878479),
 ('T1045', 0.9577574729919434),
 ('T1218.002', 0.9564635157585144),
 ('T1522', 0.9529238939285278),
 ('T1668', 0.9506741762161255),
 ('T1048', 0.9388517141342163),
 ('T1006', 0.9342100024223328),
 ('T1552.005', 0.932371199131012),
 ('T1684.002', 0.9310784339904785),
 ('T1499.001', 0.930199921131134),
 ('T1497.002', 0.9292735457420349),
 ('T1196', 0.9277694821357727),
 ('T1020.001', 0.9249854683876038),
 ('T1592.002', 0.9244982004165649),
 ('T1562.011', 0.9229199886322021),
 ('T1061', 0.9220420122146606),
 ('T1205', 0.9215130805969238),
 ('T1071.003', 0.9176769256591797),
 ('T1492', 0.9148496985435486),
 ('T1588.006', 0.9146300554275513)]

### Final R-GCN Experiment

In [50]:
import random

sample_cves = random.sample(
    cve_nodes,
    5
)

sample_cves

['CVE-2026-46932',
 'CVE-2026-46800',
 'CVE-2026-10640',
 'CVE-2026-11776',
 'CVE-2026-35312']

### Create Discovery Function

In [51]:
def discover_threats(cve_id, top_k=5):

    if cve_id not in node2id:
        print("CVE not found")
        return

    cve_idx = node2id[cve_id]

    predictions = []

    with torch.no_grad():

        cve_embedding = node_embeddings[cve_idx].unsqueeze(0)

        for attack in attack_nodes:

            attack_idx = node2id[attack]

            attack_embedding = node_embeddings[
                attack_idx
            ].unsqueeze(0)

            score = torch.sigmoid(
                predictor(
                    cve_embedding,
                    attack_embedding
                )
            ).item()

            predictions.append(
                (
                    attack,
                    score
                )
            )

    predictions = sorted(
        predictions,
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_k]

### Run on 5 CVEs

In [52]:
for cve in sample_cves:

    print("\n" + "="*70)
    print("CVE:", cve)

    results = discover_threats(
        cve,
        top_k=5
    )

    for rank, (technique, score) in enumerate(results, start=1):

        print(
            rank,
            technique,
            round(score, 4)
        )


CVE: CVE-2026-46932
1 T1020 0.9665
2 T1045 0.9643
3 T1218.002 0.9632
4 T1522 0.9601
5 T1668 0.9582

CVE: CVE-2026-46800
1 T1020 0.9504
2 T1045 0.9472
3 T1218.002 0.9456
4 T1522 0.9412
5 T1668 0.9384

CVE: CVE-2026-10640
1 T1020 0.9736
2 T1045 0.9719
3 T1218.002 0.971
4 T1522 0.9686
5 T1668 0.9671

CVE: CVE-2026-11776
1 T1020 0.9839
2 T1045 0.9828
3 T1218.002 0.9823
4 T1522 0.9808
5 T1668 0.9799

CVE: CVE-2026-35312
1 T1020 0.9594
2 T1045 0.9567
3 T1218.002 0.9554
4 T1522 0.9518
5 T1668 0.9495


### dataframe for the thesis.

In [53]:
rows = []

for cve in sample_cves:

    results = discover_threats(cve, top_k=5)

    for rank, (technique, score) in enumerate(results, start=1):

        rows.append([
            cve,
            rank,
            technique,
            score
        ])

threat_df = pd.DataFrame(
    rows,
    columns=[
        "CVE",
        "Rank",
        "Technique",
        "Score"
    ]
)

threat_df.head()

,CVE,Rank,Technique,Score
0,CVE-2026-46932,1,T1020,0.966464
1,CVE-2026-46932,2,T1045,0.964269
2,CVE-2026-46932,3,T1218.002,0.963167
3,CVE-2026-46932,4,T1522,0.960150
4,CVE-2026-46932,5,T1668,0.958230


-------------

## Saving

### Save R-GCN Model

In [54]:
import torch

model_dir = project / "models"
model_dir.mkdir(exist_ok=True)

torch.save(
    model.state_dict(),
    model_dir / "rgcn_model.pth"
)

print("R-GCN model saved")

R-GCN model saved


### Save Link Predictor

In [55]:
torch.save(
    predictor.state_dict(),
    model_dir / "rgcn_predictor.pth"
)

print("Predictor saved")

Predictor saved


### Save Node Embeddings

In [56]:
torch.save(
    node_embeddings,
    model_dir / "rgcn_node_embeddings.pt"
)

print("Node embeddings saved")

Node embeddings saved


### Save the discovery results.

In [59]:
results_dir = project / "results"

results_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = results_dir / "rgcn_threat_discovery.csv"

threat_df.to_csv(
    output_file,
    index=False
)

print("saved")

saved


### Save the Explanation Example

In [60]:
example_df = pd.DataFrame({
    "Stage":[
        "CVE",
        "Weakness",
        "Attack Pattern",
        "Technique",
        "Mitigation"
    ],
    "Value":[
        "CVE-2026-12212",
        "CWE-266",
        "CAPEC-19",
        "T1037",
        "Execution Prevention"
    ]
})

example_df

,Stage,Value
0,CVE,CVE-2026-12212
1,Weakness,CWE-266
2,Attack Pattern,CAPEC-19
3,Technique,T1037
4,Mitigation,Execution Prevention


In [61]:
example_df.to_csv(
    project /
    "results" /
    "explainable_path_example.csv",
    index=False
)

### save checkpoint

In [65]:
relations = sorted(
    all_edges_df["relation"].unique()
)

relation2id = {
    rel: idx
    for idx, rel in enumerate(relations)
}

relation2id

{'affects': 0,
 'has_weakness': 1,
 'mapped_to': 2,
 'maps_to': 3,
 'mitigated_by': 4}

In [66]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "predictor_state_dict": predictor.state_dict(),
    "node_embeddings": node_embeddings,
    "node2id": node2id,
    "relation2id": relation2id
}

torch.save(
    checkpoint,
    model_dir / "rgcn_complete_checkpoint.pt"
)

print("Complete checkpoint saved")

Complete checkpoint saved


In [67]:
test = torch.load(
    model_dir / "rgcn_complete_checkpoint.pt"
)

print(test.keys())

dict_keys(['model_state_dict', 'predictor_state_dict', 'node_embeddings', 'node2id', 'relation2id'])


### saving relation mapping

In [68]:
import pickle

with open(
    model_dir / "relation2id.pkl",
    "wb"
) as f:
    pickle.dump(
        relation2id,
        f
    )

print("relation2id saved")

relation2id saved
